# Pretraining notebook

## Section 1: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 20 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why?

In [1]:
import torch

# Reproducible results
torch.manual_seed(123)

from weird_ai.model import WeirdAIModel
from weird_ai.tokenizer import SimpleCharacterTokenizer
from weird_ai.generation import generate_and_print_sample
from weird_ai.config import SAMPLE_LYRICS_FILE


# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# Load the lyrics corpus
with open(SAMPLE_LYRICS_FILE, "r", encoding="utf-8") as f:
    text_data = f.read()


# Create the character tokenizer
tokenizer = SimpleCharacterTokenizer(text_data)

print("Vocabulary size:", len(tokenizer.chars))


# Model settings
context_length = 128
emb_dim = 128
num_heads = 4
num_layers = 2


# Create the model
model = WeirdAIModel(
    vocab_size=len(tokenizer.chars),
    context_length=context_length,
    emb_dim=emb_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=0.0,
    qkv_bias=False
)

model = model.to(device)
model.eval()


# Generate text from the three required prompts

prompts = [
    "love is",
    "baby",
    "tonight"
]

for prompt in prompts:
    print("\n" + "=" * 50)
    print("Prompt:", prompt)
    print("=" * 50)

    generate_and_print_sample(
        model=model,
        tokenizer=tokenizer,
        device=device,
        start_context=prompt,
        context_size=context_length,
        max_new_tokens=50
    )

Device: cpu
Vocabulary size: 681

Prompt: love is
love is地용마博z잘었陪든짜카这oき정들순날꺼배열éE랙으K쓰임들夜ぜ콜け老Ü만漫台到손로落底会版냥하말원내

Prompt: baby
baby中상だ위帮面짜Ő人物{벌入자館Í音﻿o순想견向n動밀
Ő냥多범잘하원이°n쓰р겨네년陪척알I쩍亮ミœ

Prompt: tonight
tonight짜빛面짜Ő범听직린入자館Í音﻿들순想견向n動밀
Ő냥多범잘하원이°n쓰р겨네년陪척알I쩍亮ミœı半и


## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities?

In [2]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum())

predicted_token = torch.argmax(probabilities)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor(1.0000)
tensor(1)


## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [3]:
import torch

# Manually calculate values

probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

target_index = 0

target_probability = probs[target_index]

print("Target probability:", target_probability)

log_probability = torch.log(target_probability)

print("Log probability:", log_probability)

loss = -log_probability

print("Manual loss:", loss)


# Compare with PyTorch

logits = torch.log(probs).unsqueeze(0)
target = torch.tensor([target_index])

torch_loss = torch.nn.functional.cross_entropy(
    logits,
    target
)

print("PyTorch loss:", torch_loss)

Target probability: tensor(0.7000)
Log probability: tensor(-0.3567)
Manual loss: tensor(0.3567)
PyTorch loss: tensor(0.3567)


## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [4]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.1825)


## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [5]:
import torch
from torch.utils.data import DataLoader

from weird_ai.dataset import LyricsDataset

train_ratio = 0.9

# Split the corpus into train and validation text

split_index = int(len(text_data) * train_ratio)

train_text = text_data[:split_index]
val_text = text_data[split_index:]

print("Training characters:", len(train_text))
print("Validation characters:", len(val_text))


# Create tokenized datasets

block_size = context_length

train_tokens = tokenizer.encode(train_text)
val_tokens = tokenizer.encode(val_text)

train_dataset = LyricsDataset(
    train_tokens,
    block_size
)

val_dataset = LyricsDataset(
    val_tokens,
    block_size
)


# Create dataloaders

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(train_loader)
print(val_loader)


# Calculate loss using a limited number of batches
# This keeps the initial-loss calculation practical on CPU.

def calculate_loss(model, data_loader, device, max_batches=10):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for x, y in data_loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1)
            )

            total_loss += loss.item()
            total_batches += 1

            if total_batches >= max_batches:
                break

    model.train()

    return total_loss / total_batches


# Calculate initial losses

initial_train_loss = calculate_loss(
    model,
    train_loader,
    device,
    max_batches=10
)

initial_val_loss = calculate_loss(
    model,
    val_loader,
    device,
    max_batches=10
)


# Record the initial losses

print("Initial training loss:", initial_train_loss)
print("Initial validation loss:", initial_val_loss)

Training characters: 12156964
Validation characters: 1350774
Initial training loss: 6.684151935577392
Initial validation loss: 6.687107229232788


## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [10]:
num_epochs = 1

# Record the loss before training
training_loss_before = initial_train_loss

# Create optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

# Train for one epoch
model.train()

total_loss = 0.0
total_batches = 0
max_training_batches = 100

for epoch in range(num_epochs):

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(x)

        # Calculate loss
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

        if total_batches >= max_training_batches:
            break


# Calculate average training loss after training
final_train_loss = total_loss / total_batches

print(f"Training loss before: {training_loss_before}")
print(f"Training loss after: {final_train_loss}")

Training loss before: 6.684151935577392
Training loss after: 2.7958069467544555


In [12]:
generate_and_print_sample(
    model=model,
    tokenizer=tokenizer,
    device=device,
    start_context="love is",
    context_size=context_length,
    max_new_tokens=20
)

love is the the the the the


## Evaluation

Did the loss decrease?

Why is decreasing loss important?

## Before and After Comparison

Prompt:

love is

Before Training:
__________________

After Training:
__________________

Reflection:

How did the output change?

What evidence do you see that the model learned something from the training data?